# Population density per municipality

This notebook processes the 2024 municipality-level dataset and calculates the natural logarithm of population density.

**Input (`data/raw/`):** `kwb2024_per_gemeente.csv`  
**Output (`data/processed/`):** `population_density_per_municipality.csv`

`population_density` is taken directly from `bevolkingsdichtheid` (inhabitants per km²) in the source data. `log_population_density` is the natural logarithm, rounded to four decimal places.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
# Resolve paths (from the notebooks folder or the repository root)
root = Path.cwd()
if not (root / "data" / "raw").is_dir():
    root = root.parent

raw_path = root / "data" / "raw" / "kwb2024_per_gemeente.csv"
output_path = root / "data" / "processed" / "population_density_per_municipality.csv"

raw = pd.read_csv(raw_path)
print("Raw rows:", len(raw))
raw.head()

In [ ]:
# Select the original population density field
population = raw[["gemeente", "bevolkingsdichtheid"]].copy()
population.columns = ["municipality", "population_density"]

# Standardize the seven municipality names used in the processed datasets
name_corrections = {
    "'s-Gravenhage": "Den Haag",
    "Beek (L.)": "Beek",
    "Hengelo (O.)": "Hengelo",
    "Laren (NH.)": "Laren",
    "Middelburg (Z.)": "Middelburg",
    "Rijswijk (ZH.)": "Rijswijk",
    "Stein (L.)": "Stein",
}
population["municipality"] = population["municipality"].replace(name_corrections)

# Calculate natural logarithm (ln) of inhabitants per km²
population["population_density"] = pd.to_numeric(
    population["population_density"], errors="raise"
)
if population["population_density"].isna().any() or (population["population_density"] <= 0).any():
    raise ValueError("Population density must be present and positive for the logarithm.")

population["log_population_density"] = np.log(
    population["population_density"]
).round(4)

# Check the data before saving
assert len(population) == 342, "Unexpected number of municipalities"
assert population["municipality"].is_unique, "Duplicate municipality names"
assert not population.isna().any().any(), "Missing values in processed data"

population.head()

In [ ]:
# Save with exactly the existing filename and column order
output_path.parent.mkdir(parents=True, exist_ok=True)
population.to_csv(output_path, index=False)
print(f"Saved {len(population)} municipalities to {output_path}")
population.head()